In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils.tools import aggregate_results

In [ ]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

In [ ]:
def heatmap(df, metric_cols):
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(16, 20))

    for i, ax in enumerate(axes):

        label, metric, *_ = metric_cols[i].split(" ")

        pivoted = df.pivot(index='model', columns='suite', values=metric_cols[i])
        pivoted = pivoted.reindex(order_suites(pivoted.columns), axis=1)
        pivoted = pivoted.reindex(order_models(pivoted.index), axis=0)
        
        ax.set_title(capitalize(label))
        
        sns.heatmap(
            pivoted,
            annot=True,
            square=True,
            cmap="Blues",
            ax=ax,
        )

        #ax.set_xticklabels(df["suite"].unique(), rotation=45)

    #plt.suptitle(f"Means of {capitalize(metric)} Scores by Label", y=1)
    plt.tight_layout()
    
    return fig, ax

In [ ]:
metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]
metric_cols_prec = ["theme precision mean", "topic precision mean", "concept precision mean"]
metric_cols_rec = ["theme recall mean", "topic recall mean", "concept recall mean"]

###
# HEATMAP F1
###
res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

fig, ax = heatmap(res, metric_cols_f1)
plt.show()
#plt.savefig("./latex/images/f1_per_suite", bbox_inches="tight")

In [ ]:
fig, ax = heatmap(res, metric_cols_prec)
#plt.savefig("./latex/images/precision_per_suite", bbox_inches="tight")

In [ ]:
fig, ax = heatmap(res, metric_cols_rec)
#plt.savefig("./latex/images/recall_per_suite", bbox_inches="tight")

In [ ]:
metric_cols = ["Theme", "Topic", "Concept", "Sum"]

f1s = res[["model", "suite"] + metric_cols_f1]
f1s = f1s.rename(columns={col: col.split(" ")[0] for col in metric_cols_f1})

f1s["sum"] = f1s["theme"] + f1s["topic"] + f1s["concept"]
f1s = f1s.rename(columns={col: capitalize(col) for col in f1s.columns})


top10s = [
    f1s
        .sort_values(by=col, ascending=False)[:10]
    for col in metric_cols
]

for df in top10s:
    print(df.to_latex(index=False, float_format="%.3f"))

In [ ]:
max_per_model = f1s.groupby('Model').idxmax()["Sum"]
tops_per_model = [
    f1s
        .loc[max_per_model]
        .sort_values(by=col, ascending=False)
    for col in metric_cols
]

for df in tops_per_model:
    print(df.to_latex(index=False, float_format="%.3f"))